# Certifying a decay rate: damped pendulum

This notebook certifies an exponential decay rate for a damped pendulum on the
box `‖x‖∞ ≤ 0.8` with `verify_stability`. The vector field is written with NumPy
(`np.sin`), so pyDDRV uses its NumPy kernel and JAX is not needed. The Lipschitz
constant is estimated with the extreme-value method, because the Jacobian of
the pendulum is not affine in the state.

Requirements: Python 3.10 or later and
`pip install "pyddrv[examples] @ git+https://github.com/NetDLab/pyDDRV"`.
See `examples/notebooks/README.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyddrv import verify_stability
from pyddrv.systems import damped_pendulum

## The system

The state is `x = (θ, ω)` and

$$\dot\theta = \omega, \qquad \dot\omega = -\sin\theta - \omega.$$

pyDDRV expects a batched field: it maps an array of states of shape `(N, d)` to
an array of derivatives of the same shape.

In [ ]:
f = damped_pendulum(g_over_l=1.0, damping=1.0)   # batched field (N,2) -> (N,2)
f(np.array([[0.3, -0.1], [0.0, 0.5]]))           # quick sanity evaluation

## The Lipschitz constant

The certificate needs an upper bound `L` on the one-sided Lipschitz constant of
the field over the states visited by trajectories from the box. The Jacobian of
the pendulum is

$$\frac{\partial f}{\partial x} = \begin{bmatrix} 0 & 1 \\ -\cos\theta & -1 \end{bmatrix}.$$

Its matrix measure depends on `θ` through `cos θ`, so its maximum over a box is
not always at a corner: for boxes that contain `θ = π` the maximum is inside
the box. With `R = 0.8` the reachable set does not reach `θ = π` and the corners
would happen to give the right value, but that is only known from the analysis
above. The general method is the extreme-value estimator, which samples the
inside of the box and returns a bound that holds with probability `rho`. With
no Jacobian supplied, the default `L_method="auto"` makes this choice; the code
below passes `L_method="evt"` explicitly.

For this field a closed-form bound exists: the matrix measure never exceeds
`(√5 − 1)/2 ≈ 0.618`. Passing `L=0.62` instead makes the certificate
deterministic.

In [ ]:
report = verify_stability(f, R=0.8, d=2, tau=6.0,
                          L_method="evt", rho=0.95,
                          delta=0.2, max_refine=6, record_trace=True)
print(report.summary())

## The report

`report.certified` is true when a positive rate was certified and the
discretization check passed. `report.alpha` is the certified rate, and
`report.alpha_upper` is the rate certified at the cube centers alone.
`report.lipschitz.evt` describes the extreme-value fit: the fitted endpoint
`gamma`, the Kolmogorov-Smirnov p-value, and whether the fit was accepted.

In [ ]:
print("certified        :", report.certified)
print("alpha (lower bnd):", round(report.alpha, 4))
print("alpha_upper      :", round(report.alpha_upper, 4))
print("L (one-sided)    :", round(report.L, 4))
print("discretization   :", report.discretization_ok)
print("EVT:", report.lipschitz.evt.summary())

## Plots

`plot_stability_2d` draws the certified box over the phase portrait.
`plot_anytime` shows the certified rate after each refinement round against
wall time. Each point is a valid lower bound, with probability `rho` since `L`
is estimated.

In [ ]:
from pyddrv.viz import plot_stability_2d, plot_anytime

ax = plot_stability_2d(report, f=f)
ax.set_xlabel(r"$\theta$"); ax.set_ylabel(r"$\omega$")
ax.set_title("Damped pendulum: certified exponential decay on $Q_R$")
plt.show()

In [ ]:
ax = plot_anytime(report)
ax.set_title("Anytime certified rate")
plt.show()

### The cubes used by the certificate

With `show_grid=True`, `plot_stability_2d` draws the cubes that cover the box:
as outlines over the flow, filled by their width, or filled by the rate each
cube certifies. Cubes are large far from the equilibrium and smaller near it,
and refinement adds smaller cubes where the rate is limited. The cubes with the
lowest rate determine `report.alpha`.

In [ ]:
plot_stability_2d(report, f=f, show_grid=True, grid_color_by="outline")
plt.title("Certified covering grid over the flow"); plt.show()

plot_stability_2d(report, show_grid=True, grid_color_by="width")
plt.title("Cube widths (log scale)"); plt.show()

plot_stability_2d(report, show_grid=True, grid_color_by="alpha")
plt.title("Per-cube certified rate"); plt.show()